# Reproducibility smoke test: 500-patient sample

This notebook runs `../validation/benchmark_script.py`, **unmodified, the exact script used for the published benchmark**, against the 500-patient sample in `../sample/data` (scenarios S10, S13, S16; the script defaults to scenario S12, so here it is pointed at S13, the middle scenario of this sample).

**Purpose:** show the code runs end to end against the real delivered schema (partitioned `observed/` and `truth/` tables, control world W0).

**Not the purpose:** reproduce the published metrics. The full-dataset benchmark (n = 10,000) reports AUC 0.697 (admission only) to 0.871 (cultures to day 12) — see the dataset README. This sample has only 500 patients, so the numbers below are for a code check, not a number to cite.

In [1]:
import subprocess, sys
r = subprocess.run([sys.executable, "../validation/benchmark_script.py", "../sample/data", "./smoke_test_results.parquet", "S13"],
                   capture_output=True, text=True)
print(r.stdout[-3000:])
print(r.stderr[-2000:])
assert r.returncode == 0, r.stderr

NaN                          NaN  424.0                  NaN       NaN                                    NaN                               NaN
T4_landmark_day12_screen_negative                 admission_data_plus_cultures_resulted_by_day12                           logistic_regression       NaN     0.918780    0.852436    0.884747                        NaN                          NaN  424.0                  NaN       NaN                                    NaN                               NaN
                                                                                                                           xgboost                   NaN     0.923083    0.856832    0.889246                        NaN                          NaN  424.0                  NaN       NaN                                    NaN                               NaN
T4_resistant_culture_on_day14_screen_negative     admission_data_plus_cultures_resulted_by_day9                            logistic_regression

## Reading the output

- `n` for each task is around 500 (one sampled scenario, one world), far below the 10,000-patient cohort the published numbers come from, so AUCs here are noisy and must not be quoted.
- Tasks T1–T5 and the landmark rows follow the same definitions as the published benchmark (see the dataset README and passport).
- The published, citable numbers are in `../validation/benchmark_results.parquet` (full 10,000-patient cohort) and in the README.

In [2]:
import pandas as pd
res = pd.read_parquet("./smoke_test_results.parquet")
res[res.metric.isin(["auc_pooled", "auc_ci_low", "auc_ci_high", "n"])].pivot_table(
    index=["task", "model"], columns="metric", values="value", aggfunc="first")

metric                                                                 auc_ci_high  \
task                                              model                              
T1_new_resistant_culture_by_day14                 logistic_regression     0.747164   
                                                  xgboost                 0.708021   
T2_new_resistant_culture_by_day14_screen_negative logistic_regression     0.816353   
                                                  xgboost                 0.813421   
T3_landmark_day02_resistant_culture_on_day14      logistic_regression     0.774211   
                                                  xgboost                 0.748168   
T3_landmark_day04_resistant_culture_on_day14      logistic_regression     0.801615   
                                                  xgboost                 0.794179   
T3_landmark_day12_resistant_culture_on_day14      logistic_regression     0.893679   
                                                  xgboost                 0.901973   
T3_resistant_culture_on_day14_whole_cohort        logistic_regression     0.856976   
                                                  xgboost                 0.863775   
T4_landmark_day02_screen_negative                 logistic_regression     0.787897   
                                                  xgboost                 0.756213   
T4_landmark_day04_screen_negative                 logistic_regression     0.837924   
                                                  xgboost                 0.810318   
T4_landmark_day12_screen_negative                 logistic_regression     0.918780   
                                                  xgboost                 0.923083   
T4_resistant_culture_on_day14_screen_negative     logistic_regression     0.892647   
                                                  xgboost                 0.879262   
T5_death_by_day90                                 logistic_regression     0.572769   
                                                  xgboost                 0.618725   

metric                                                                 auc_ci_low  \
task                                              model                             
T1_new_resistant_culture_by_day14                 logistic_regression    0.662871   
                                                  xgboost                0.616478   
T2_new_resistant_culture_by_day14_screen_negative logistic_regression    0.730428   
                                                  xgboost                0.725819   
T3_landmark_day02_resistant_culture_on_day14      logistic_regression    0.692879   
                                                  xgboost                0.656438   
T3_landmark_day04_resistant_culture_on_day14      logistic_regression    0.718833   
                                                  xgboost                0.711249   
T3_landmark_day12_resistant_culture_on_day14      logistic_regression    0.829634   
                                                  xgboost                0.839924   
T3_resistant_culture_on_day14_whole_cohort        logistic_regression    0.780950   
                                                  xgboost                0.788201   
T4_landmark_day02_screen_negative                 logistic_regression    0.696003   
                                                  xgboost                0.666544   
T4_landmark_day04_screen_negative                 logistic_regression    0.746963   
                                                  xgboost                0.722650   
T4_landmark_day12_screen_negative                 logistic_regression    0.852436   
                                                  xgboost                0.856832   
T4_resistant_culture_on_day14_screen_negative     logistic_regression    0.809029   
                                                  xgboost                0.803445   
T5_death_by_day90                                 logistic_regres